# Laboratório 1 – Tarefa 2: Web Scraping em Ambiente Real (IMDb)

**Integrantes:** Gabrielle Guarani da Silva e Luiza Hackeenhar Naziazeno

**Disciplina:** Coleta, Preparação e Análise de Dados  

**Professora:** Katherine Bianchini Esper


## 1. Importação das bibliotecas

In [ ]:
# Controle do navegador via Selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

# Utilitários
import json
import time
import re
import base64
import requests

print("Bibliotecas importadas com sucesso!")

: 

## 2. Inicialização do WebDriver

Configuramos o Chrome em modo **headless** (sem abrir janela visível), o que é mais eficiente para coleta de dados.  
Adicionamos também um `user-agent` para simular um navegador real e evitar bloqueios automáticos.

In [ ]:
# Configurações do Chrome
opcoes = Options()

# Remove a janela visível do navegador (mais rápido e silencioso)
opcoes.add_argument("--headless=new")

# Evita erros de sandbox em ambientes Linux
opcoes.add_argument("--no-sandbox")
opcoes.add_argument("--disable-dev-shm-usage")

# Simula um navegador real para reduzir bloqueios (para esta parte utilizamos IA)
opcoes.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36"
)

# Inicia o driver (o selenium-manager baixa o chromedriver automaticamente)
driver = webdriver.Chrome(options=opcoes)

# Espera inteligente: aguarda até 15 segundos por elementos na página
wait = WebDriverWait(driver, 15)

print("WebDriver iniciado com sucesso!")

## 3. Etapa 1 — Coleta da lista Top 250

Acessamos a página `https://www.imdb.com/pt/chart/top/`, que contém os 250 filmes mais bem avaliados.  
Para cada filme da lista, coletamos:
- **Título**
- **Ano de lançamento** (extraído da metadata do card)
- **Nota IMDb**
- **URL da página individual** do filme

In [ ]:
# Acessa a página Top 250 do IMDb
URL_TOP250 = "https://www.imdb.com/pt/chart/top/"
driver.get(URL_TOP250)

# Aguarda a lista de filmes aparecer no DOM
wait.until(
    EC.presence_of_element_located((
        By.CSS_SELECTOR,
        "li.ipc-metadata-list-summary-item"
    ))
)

# Pequena pausa para garantir que os elementos dinâmicos carregaram
time.sleep(2)

# Coleta todos os cards de filmes da lista
cards = driver.find_elements(By.CSS_SELECTOR, "li.ipc-metadata-list-summary-item")

print(f"Total de filmes encontrados na lista: {len(cards)}")

In [ ]:
# Extrai dados básicos de cada card da lista
lista_filmes = []   # Armazenará dicionários com dados básicos mais a URL de cada filme

for card in cards:

    # --- Título ---
    try:
        titulo = card.find_element(By.CSS_SELECTOR, "h3").text.strip()
        # O IMDb costuma prefixar com número de ranking (ex: "1. Um Sonho de Liberdade")
        # Removemos o prefixo numérico, se existir
        titulo = re.sub(r"^\d+\.\s*", "", titulo)
    except Exception:
        titulo = None

    # --- Ano de lançamento ---
    # O ano fica em um <li> com texto de 4 dígitos dentro dos metadados do card
    ano = None
    try:
        meta_items = card.find_elements(By.CSS_SELECTOR, "li.ipc-inline-list__item")
        for item in meta_items:
            texto = item.text.strip()
            if re.fullmatch(r"\d{4}", texto):
                ano = texto
                break
    except Exception:
        ano = None

    # --- Nota IMDb ---
    try:
        nota = card.find_element(By.CSS_SELECTOR, "span.ipc-rating-star").text.strip()
        # A nota pode vir como "9,3" ou "9.3" dependendo do locale; normalizamos
        nota = nota.split()[0].replace(",", ".")
    except Exception:
        nota = None

    # --- URL da página do filme ---
    try:
        url_filme = card.find_element(By.CSS_SELECTOR, "a.ipc-title-link-wrapper").get_attribute("href")
    except Exception:
        try:
            # Fallback: pega o primeiro link <a> do card
            url_filme = card.find_element(By.TAG_NAME, "a").get_attribute("href")
        except Exception:
            url_filme = None

    lista_filmes.append({
        "titulo": titulo,
        "ano": ano,
        "nota_imdb": nota,
        "url_filme": url_filme
    })

# Filtra entradas sem URL (inválidas)
lista_filmes = [f for f in lista_filmes if f["url_filme"]]

print(f"Filmes com URL válida: {len(lista_filmes)}")
print("\nPrimeiros 3 registros da lista:")
for f in lista_filmes[:3]:
    print(f)

: 

## 4. Etapa 2 — Scraping das páginas individuais

Para cada filme da lista, abrimos a URL da página individual em uma nova aba.
Coletamos as informações adicionais:
- **URL do pôster** e **imagem do pôster** (base64)
- **Lista de gêneros**
- **Lista de direção**

> ⚠️ **Atenção:** Adicionamos uma pausa de 1 segundo entre cada requisição para não sobrecarregar o servidor do IMDb.

In [ ]:
def baixar_imagem_base64(url_imagem: str) -> str | None:
    """
    Baixa a imagem de uma URL e retorna os bytes codificados em base64.
    Retorna None em caso de falha.
    """
    if not url_imagem:
        return None
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                          "AppleWebKit/537.36 (KHTML, like Gecko) "
                          "Chrome/124.0.0.0 Safari/537.36"
        }
        resp = requests.get(url_imagem, headers=headers, timeout=10)
        resp.raise_for_status()
        return base64.b64encode(resp.content).decode("utf-8")
    except Exception:
        return None

def scrape_pagina_filme(driver, wait, url: str) -> dict:
    """
    Abre a página do filme em uma nova aba, extrai os dados e fecha a aba.
    Retorna um dicionário com: poster_url, poster_base64, generos, direcao.
    """
    aba_principal = driver.current_window_handle

    # Abre a página do filme em nova aba
    driver.execute_script("window.open(arguments[0]);", url)
    driver.switch_to.window(driver.window_handles[-1])

    dados_pagina = {
        "poster_url": None,
        "poster_base64": None,
        "generos": [],
        "direcao": []
    }

    try:
        # Espera o título principal aparecer (garante que a página carregou)
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
        time.sleep(1)  # Pausa adicional para elementos dinâmicos

        # --- URL do pôster ---
        try:
            poster_img = driver.find_element(
                By.CSS_SELECTOR,
                "div.ipc-media--poster-l img, "
                "div.ipc-media--poster-m img, "
                "div.ipc-poster__poster-image img"
            )
            dados_pagina["poster_url"] = poster_img.get_attribute("src")
        except Exception:
            dados_pagina["poster_url"] = None

        # --- Imagem do pôster em base64 ---
        dados_pagina["poster_base64"] = baixar_imagem_base64(dados_pagina["poster_url"])

        # --- Gêneros ---
        try:
            genero_elements = driver.find_elements(
                By.CSS_SELECTOR,
                "div.ipc-chip-list--baseAlt a.ipc-chip span.ipc-chip__text, "
                "a[href*='/search/title/?genres='] span"
            )
            generos = [g.text.strip() for g in genero_elements if g.text.strip()]
            # Remove duplicatas mantendo a ordem
            vistos = set()
            dados_pagina["generos"] = [
                g for g in generos if not (g in vistos or vistos.add(g))
            ]
        except Exception:
            dados_pagina["generos"] = []

        # --- Direção (diretores) ---
        direcao = []
        try:
            # Estratégia 1: procurar o bloco de crédito que contém "Direção" ou "Director"
            blocos_credito = driver.find_elements(
                By.CSS_SELECTOR,
                "div[data-testid='title-pc-principal-credit'], li.ipc-metadata-list__item"
            )
            for bloco in blocos_credito:
                texto_bloco = bloco.text.lower()
                if "direção" in texto_bloco or "director" in texto_bloco:
                    links = bloco.find_elements(By.CSS_SELECTOR, "a")
                    direcao = [link.text.strip() for link in links if link.text.strip()]
                    if direcao:
                        break

            # Estratégia 2 (fallback): usar XPath direto por texto
            if not direcao:
                try:
                    elemento_label = driver.find_element(
                        By.XPATH,
                        "//*[contains(text(),'Direção') or contains(text(),'Director')]"
                    )
                    container = elemento_label.find_element(
                        By.XPATH,
                        "./ancestor::li[contains(@class,'ipc-metadata-list__item')] | "
                        "./ancestor::div[contains(@data-testid,'credit')]"
                    )
                    links = container.find_elements(By.CSS_SELECTOR, "a")
                    direcao = [link.text.strip() for link in links if link.text.strip()]
                except Exception:
                    pass
        except Exception:
            pass

        dados_pagina["direcao"] = direcao

    except Exception as e:
        print(f"    [ERRO ao processar página] {e}")

    finally:
        # Sempre fecha a aba extra e volta para a aba principal
        driver.close()
        driver.switch_to.window(aba_principal)

    return dados_pagina

print("Funções auxiliares definidas.")

In [ ]:
# Loop principal: visita cada página de filme e enriquece os dados
# IMPORTANTE: inserimos uma pausa de 1 segundo entre requisições para respeitar o servidor do IMDb e evitar bloqueios por excesso de requisições.

dados_completos = []   # Lista final com todos os dados de cada filme

total = len(lista_filmes)

for i, filme in enumerate(lista_filmes):

    print(f"[{i+1}/{total}] Processando: {filme['titulo']} ...")

    # Coleta dados da página individual
    dados_pagina = scrape_pagina_filme(driver, wait, filme["url_filme"])

    # Monta o dicionário final do filme
    dados_completos.append({
        "titulo":        filme["titulo"],
        "ano":           filme["ano"],
        "nota_imdb":     filme["nota_imdb"],
        "url_filme":     filme["url_filme"],
        "poster_url":    dados_pagina["poster_url"],
        "poster_base64": dados_pagina["poster_base64"],   # None se não baixou
        "generos":       dados_pagina["generos"],         # [] se não encontrou
        "direcao":     dados_pagina["direcao"]        # [] se não encontrou
    })

    # Pausa entre requisições
    time.sleep(1)

print(f"\nColeta finalizada! Total de filmes processados: {len(dados_completos)}")

: 

In [ ]:
## 5. Encerramento do WebDriver

In [ ]:
# Fecha o navegador após a coleta
driver.quit()
print("WebDriver encerrado.")

## 6. Verificação dos dados coletados

Exibimos um resumo para conferir a qualidade da coleta antes de salvar.

In [ ]:
# Estatísticas de cobertura
total = len(dados_completos)

sem_titulo    = sum(1 for f in dados_completos if not f["titulo"])
sem_ano       = sum(1 for f in dados_completos if not f["ano"])
sem_nota      = sum(1 for f in dados_completos if not f["nota_imdb"])
sem_poster    = sum(1 for f in dados_completos if not f["poster_url"])
sem_img       = sum(1 for f in dados_completos if not f["poster_base64"])
sem_generos   = sum(1 for f in dados_completos if not f["generos"])
sem_direcao = sum(1 for f in dados_completos if not f["direcao"])

print(f"Total de filmes coletados : {total}")
print(f"Sem título                : {sem_titulo}")
print(f"Sem ano                   : {sem_ano}")
print(f"Sem nota IMDb             : {sem_nota}")
print(f"Sem URL do pôster         : {sem_poster}")
print(f"Sem imagem do pôster      : {sem_img}")
print(f"Sem gêneros               : {sem_generos}")
print(f"Sem direção             : {sem_direcao}")

print("\n── Primeiros 3 filmes ──")
for f in dados_completos[:3]:
    # Exibe o base64 truncado para não poluir a saída
    resumo = dict(f)
    if resumo["poster_base64"]:
        resumo["poster_base64"] = resumo["poster_base64"][:40] + "...[base64 truncado]"
    print(json.dumps(resumo, indent=2, ensure_ascii=False))
    print()

## 7. Exportação para JSON

Salvamos os dados completos (incluindo o base64 do pôster) em `imdb_top250.json`.

In [ ]:
ARQUIVO_JSON = "imdb_top250.json"

with open(ARQUIVO_JSON, "w", encoding="utf-8") as f:
    json.dump(dados_completos, f, indent=2, ensure_ascii=False)

print(f"Arquivo '{ARQUIVO_JSON}' salvo com sucesso!")
print(f"Total de registros: {len(dados_completos)}")

## 8. Versão sem imagens em base64

Se o arquivo JSON ficou muito grande, esta célula salva uma versão mais leve sem o campo `poster_base64`.

In [ ]:
ARQUIVO_JSON_LEVE = "imdb_top250_sem_imagens.json"

# Copia os dados omitindo o campo poster_base64
dados_leve = [
    {k: v for k, v in filme.items() if k != "poster_base64"}
    for filme in dados_completos
]

with open(ARQUIVO_JSON_LEVE, "w", encoding="utf-8") as f:
    json.dump(dados_leve, f, indent=2, ensure_ascii=False)

print(f"Arquivo leve '{ARQUIVO_JSON_LEVE}' salvo com sucesso!")

---

## Estrutura do JSON gerado

Um exemplo, cada entrada do arquivo JSON tem o seguinte formato:

```json
{
  "titulo": "Um Sonho de Liberdade",
  "ano": "1994",
  "nota_imdb": "9.3",
  "url_filme": "https://www.imdb.com/title/tt0111161/",
  "poster_url": "https://m.media-amazon.com/images/...",
  "poster_base64": "<bytes da imagem em base64>",
  "generos": ["Drama"],
  "direcao": ["Frank Darabont"]
}
```

Campos ausentes (não encontrados na página) são representados como `null` (para strings) ou `[]` (para listas).